# Spark Database Integration - PostgreSQL & External Data Sources

This notebook demonstrates how to integrate Apache Spark with external databases and data sources:
- Reading from and writing to PostgreSQL
- JDBC connectivity and optimization
- Data lake integration patterns
- ETL/ELT workflows
- Performance optimization for database operations
- Handling large datasets with partitioning

## Setup and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# Create Spark Session with database connectivity
spark = SparkSession.builder \
    .appName("Database Integration Demo") \
    .master("spark://spark-master:7077") \
    .config("spark.jars", "/opt/bitnami/spark/jars/postgresql-42.7.1.jar") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Database connection properties
db_properties = {
    "url": "jdbc:postgresql://postgres:5432/sparkdb",
    "user": "spark",
    "password": "sparkpassword",
    "driver": "org.postgresql.Driver"
}

print(f"Spark Version: {spark.version}")
print("Database integration environment ready!")

## 1. Basic Database Connectivity

In [ ]:
# Test database connection by reading existing tables
try:
    # Read the sales_summary table that was created during setup
    sales_summary_df = spark.read \
        .format("jdbc") \
        .option("url", db_properties["url"]) \
        .option("dbtable", "sales_summary") \
        .option("user", db_properties["user"]) \
        .option("password", db_properties["password"]) \
        .option("driver", db_properties["driver"]) \
        .load()
    
    print("✓ Successfully connected to PostgreSQL")
    print(f"Sales summary records: {sales_summary_df.count()}")
    sales_summary_df.show()
    
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    print("Make sure PostgreSQL container is running and accessible")

## 2. Reading Data from Multiple Sources

In [ ]:
# Load CSV data for comparison and processing
sales_csv = spark.read.csv("/home/jovyan/data/csv/sales_data.csv", header=True, inferSchema=True)
customers_csv = spark.read.csv("/home/jovyan/data/csv/customers.csv", header=True, inferSchema=True)
products_csv = spark.read.csv("/home/jovyan/data/csv/products.csv", header=True, inferSchema=True)

print("Data sources loaded:")
print(f"• Sales CSV: {sales_csv.count()} records")
print(f"• Customers CSV: {customers_csv.count()} records")
print(f"• Products CSV: {products_csv.count()} records")

# Create temporary views for easier querying
sales_csv.createOrReplaceTempView("sales_csv")
customers_csv.createOrReplaceTempView("customers_csv")
products_csv.createOrReplaceTempView("products_csv")

## 3. Writing Data to PostgreSQL

In [ ]:
# Create customer analytics summary for database storage
customer_analytics = spark.sql("""
    SELECT 
        c.customer_id,
        c.first_name,
        c.last_name,
        c.age,
        c.city,
        c.loyalty_tier,
        COUNT(s.transaction_id) as total_orders,
        ROUND(SUM(s.total_amount), 2) as lifetime_value,
        ROUND(AVG(s.total_amount), 2) as avg_order_value,
        COUNT(DISTINCT s.category) as categories_purchased,
        MAX(s.date) as last_purchase_date,
        CURRENT_TIMESTAMP() as analysis_timestamp
    FROM customers_csv c
    LEFT JOIN sales_csv s ON c.customer_id = s.customer_id
    GROUP BY c.customer_id, c.first_name, c.last_name, c.age, c.city, c.loyalty_tier
    HAVING COUNT(s.transaction_id) > 0
    ORDER BY lifetime_value DESC
""")

print(f"Customer analytics prepared: {customer_analytics.count()} customers")
customer_analytics.show(10)

In [ ]:
# Write customer analytics to PostgreSQL
try:
    customer_analytics.write \
        .format("jdbc") \
        .option("url", db_properties["url"]) \
        .option("dbtable", "customer_analytics") \
        .option("user", db_properties["user"]) \
        .option("password", db_properties["password"]) \
        .option("driver", db_properties["driver"]) \
        .mode("overwrite") \
        .save()
    
    print("✓ Customer analytics written to PostgreSQL")
    
except Exception as e:
    print(f"❌ Failed to write to database: {e}")

## 4. Advanced JDBC Operations with Custom Queries

In [ ]:
# Read data using custom SQL query
custom_query = """
    (SELECT 
        customer_id,
        first_name || ' ' || last_name as full_name,
        age,
        loyalty_tier,
        lifetime_value,
        avg_order_value,
        CASE 
            WHEN lifetime_value >= 3000 THEN 'High Value'
            WHEN lifetime_value >= 1500 THEN 'Medium Value'
            ELSE 'Low Value'
        END as value_segment
     FROM customer_analytics
     WHERE lifetime_value > 500
     ORDER BY lifetime_value DESC
    ) as customer_segments
"""

customer_segments_df = spark.read \
    .format("jdbc") \
    .option("url", db_properties["url"]) \
    .option("dbtable", custom_query) \
    .option("user", db_properties["user"]) \
    .option("password", db_properties["password"]) \
    .option("driver", db_properties["driver"]) \
    .load()

print("Customer segments from database:")
customer_segments_df.show(10)

# Analyze segments
segment_analysis = customer_segments_df.groupBy("value_segment") \
    .agg(
        count("*").alias("customer_count"),
        round(avg("lifetime_value"), 2).alias("avg_lifetime_value"),
        round(avg("avg_order_value"), 2).alias("avg_order_value"),
        round(avg("age"), 1).alias("avg_age")
    ) \
    .orderBy(desc("avg_lifetime_value"))

print("\nSegment analysis:")
segment_analysis.show()

## 5. Partitioned Reads for Large Datasets

In [ ]:
# Create a larger dataset for partitioning demonstration
# First, let's create a product sales summary table
product_sales_summary = spark.sql("""
    SELECT 
        p.product_id,
        p.category,
        p.product_name,
        p.cost_price,
        p.suggested_price,
        COUNT(s.transaction_id) as units_sold,
        ROUND(SUM(s.total_amount), 2) as total_revenue,
        ROUND(SUM(s.quantity * p.cost_price), 2) as total_cost,
        ROUND(AVG(s.total_amount), 2) as avg_sale_amount,
        COUNT(DISTINCT s.customer_id) as unique_customers,
        ROUND(
            (SUM(s.total_amount) - SUM(s.quantity * p.cost_price)) / SUM(s.total_amount) * 100, 2
        ) as profit_margin_pct,
        CURRENT_TIMESTAMP() as created_at
    FROM products_csv p
    JOIN sales_csv s ON p.product_name = s.product_name AND p.category = s.category
    GROUP BY p.product_id, p.category, p.product_name, p.cost_price, p.suggested_price
""")

print(f"Product sales summary: {product_sales_summary.count()} products")
product_sales_summary.show(10)

In [ ]:
# Write product summary with better performance options
try:
    product_sales_summary.write \
        .format("jdbc") \
        .option("url", db_properties["url"]) \
        .option("dbtable", "product_sales_summary") \
        .option("user", db_properties["user"]) \
        .option("password", db_properties["password"]) \
        .option("driver", db_properties["driver"]) \
        .option("batchsize", "1000") \
        .option("numPartitions", "4") \
        .mode("overwrite") \
        .save()
    
    print("✓ Product sales summary written to PostgreSQL")
    
except Exception as e:
    print(f"❌ Failed to write product summary: {e}")

In [ ]:
# Demonstrate partitioned reading for better performance
# Read product data with partitioning
partitioned_products = spark.read \
    .format("jdbc") \
    .option("url", db_properties["url"]) \
    .option("dbtable", "product_sales_summary") \
    .option("user", db_properties["user"]) \
    .option("password", db_properties["password"]) \
    .option("driver", db_properties["driver"]) \
    .option("partitionColumn", "units_sold") \
    .option("lowerBound", "1") \
    .option("upperBound", "1000") \
    .option("numPartitions", "4") \
    .load()

print(f"Partitioned read completed: {partitioned_products.count()} products")
print(f"Number of partitions: {partitioned_products.rdd.getNumPartitions()}")

# Show partition distribution
partitioned_products.groupBy(spark_partition_id().alias("partition_id")) \
    .count() \
    .orderBy("partition_id") \
    .show()

## 6. ETL Pipeline with Multiple Data Sources

In [ ]:
# Create a comprehensive ETL pipeline that combines multiple sources
def create_unified_sales_report():
    """Create a unified sales report combining CSV and database sources"""
    
    # Extract: Read from multiple sources
    print("📥 EXTRACT: Reading from multiple sources...")
    
    # CSV sources
    sales_data = spark.read.csv("/home/jovyan/data/csv/sales_data.csv", header=True, inferSchema=True)
    customers_data = spark.read.csv("/home/jovyan/data/csv/customers.csv", header=True, inferSchema=True)
    
    # Database sources
    db_customer_analytics = spark.read \
        .format("jdbc") \
        .option("url", db_properties["url"]) \
        .option("dbtable", "customer_analytics") \
        .option("user", db_properties["user"]) \
        .option("password", db_properties["password"]) \
        .option("driver", db_properties["driver"]) \
        .load()
    
    print(f"  • Sales data: {sales_data.count()} records")
    print(f"  • Customer analytics from DB: {db_customer_analytics.count()} records")
    
    # Transform: Create comprehensive sales report
    print("🔄 TRANSFORM: Processing and enriching data...")
    
    sales_data.createOrReplaceTempView("sales")
    customers_data.createOrReplaceTempView("customers")
    db_customer_analytics.createOrReplaceTempView("customer_analytics_db")
    
    unified_report = spark.sql("""
        WITH enriched_sales AS (
            SELECT 
                s.*,
                c.age,
                c.city,
                c.loyalty_tier,
                ca.lifetime_value,
                ca.avg_order_value as customer_avg_order,
                ca.total_orders as customer_total_orders,
                CASE 
                    WHEN ca.lifetime_value >= 3000 THEN 'High Value'
                    WHEN ca.lifetime_value >= 1500 THEN 'Medium Value'
                    ELSE 'Low Value'
                END as customer_value_segment,
                MONTH(s.date) as sale_month,
                DAYOFWEEK(s.date) as sale_day_of_week,
                QUARTER(s.date) as sale_quarter
            FROM sales s
            LEFT JOIN customers c ON s.customer_id = c.customer_id
            LEFT JOIN customer_analytics_db ca ON s.customer_id = ca.customer_id
        )
        SELECT 
            -- Transaction details
            transaction_id,
            customer_id,
            date as transaction_date,
            category,
            product_name,
            quantity,
            unit_price,
            total_amount,
            region,
            
            -- Customer details
            age,
            city,
            loyalty_tier,
            customer_value_segment,
            
            -- Customer analytics
            lifetime_value,
            customer_avg_order,
            customer_total_orders,
            
            -- Time dimensions
            sale_month,
            sale_day_of_week,
            sale_quarter,
            
            -- Derived metrics
            ROUND(total_amount / customer_avg_order, 2) as transaction_vs_avg_ratio,
            CASE 
                WHEN sale_day_of_week IN (1, 7) THEN 'Weekend'
                ELSE 'Weekday'
            END as weekend_flag,
            
            CURRENT_TIMESTAMP() as etl_timestamp
        FROM enriched_sales
        WHERE lifetime_value IS NOT NULL
    """)
    
    print(f"  • Unified report created: {unified_report.count()} enriched records")
    
    return unified_report

# Execute ETL pipeline
unified_sales_report = create_unified_sales_report()
unified_sales_report.show(10)

In [ ]:
# Load: Write the unified report back to database
print("💾 LOAD: Writing unified report to database...")

try:
    unified_sales_report.write \
        .format("jdbc") \
        .option("url", db_properties["url"]) \
        .option("dbtable", "unified_sales_report") \
        .option("user", db_properties["user"]) \
        .option("password", db_properties["password"]) \
        .option("driver", db_properties["driver"]) \
        .option("batchsize", "1000") \
        .option("numPartitions", "8") \
        .mode("overwrite") \
        .save()
    
    print("✅ ETL Pipeline completed successfully!")
    print("   Unified sales report available in PostgreSQL")
    
except Exception as e:
    print(f"❌ ETL Load phase failed: {e}")

# Verify the load by reading a sample
verification_query = """
    (SELECT 
        customer_value_segment,
        COUNT(*) as transaction_count,
        ROUND(AVG(total_amount), 2) as avg_transaction_amount,
        ROUND(SUM(total_amount), 2) as total_revenue
     FROM unified_sales_report 
     GROUP BY customer_value_segment
     ORDER BY total_revenue DESC
    ) as segment_summary
"""

verification_df = spark.read \
    .format("jdbc") \
    .option("url", db_properties["url"]) \
    .option("dbtable", verification_query) \
    .option("user", db_properties["user"]) \
    .option("password", db_properties["password"]) \
    .option("driver", db_properties["driver"]) \
    .load()

print("\n📊 Verification - Customer segment performance:")
verification_df.show()

## 7. Database Performance Optimization

In [ ]:
# Demonstrate different performance optimization techniques

def benchmark_read_methods():
    """Compare different database read methods"""
    
    print("🚀 Performance Benchmarking:")
    
    # Method 1: Standard read
    start_time = time.time()
    standard_read = spark.read \
        .format("jdbc") \
        .option("url", db_properties["url"]) \
        .option("dbtable", "customer_analytics") \
        .option("user", db_properties["user"]) \
        .option("password", db_properties["password"]) \
        .option("driver", db_properties["driver"]) \
        .load()
    
    standard_count = standard_read.count()
    standard_time = time.time() - start_time
    
    print(f"  📋 Standard read: {standard_count} records in {standard_time:.2f}s")
    
    # Method 2: Partitioned read
    start_time = time.time()
    partitioned_read = spark.read \
        .format("jdbc") \
        .option("url", db_properties["url"]) \
        .option("dbtable", "customer_analytics") \
        .option("user", db_properties["user"]) \
        .option("password", db_properties["password"]) \
        .option("driver", db_properties["driver"]) \
        .option("partitionColumn", "age") \
        .option("lowerBound", "18") \
        .option("upperBound", "80") \
        .option("numPartitions", "4") \
        .load()
    
    partitioned_count = partitioned_read.count()
    partitioned_time = time.time() - start_time
    
    print(f"  ⚡ Partitioned read: {partitioned_count} records in {partitioned_time:.2f}s")
    
    # Method 3: Custom query with filtering
    start_time = time.time()
    filtered_query = """
        (SELECT * FROM customer_analytics 
         WHERE lifetime_value > 1000 
         AND total_orders >= 5
        ) as filtered_customers
    """
    
    filtered_read = spark.read \
        .format("jdbc") \
        .option("url", db_properties["url"]) \
        .option("dbtable", filtered_query) \
        .option("user", db_properties["user"]) \
        .option("password", db_properties["password"]) \
        .option("driver", db_properties["driver"]) \
        .load()
    
    filtered_count = filtered_read.count()
    filtered_time = time.time() - start_time
    
    print(f"  🎯 Filtered read: {filtered_count} records in {filtered_time:.2f}s")
    
    return {
        "standard": (standard_count, standard_time),
        "partitioned": (partitioned_count, partitioned_time),
        "filtered": (filtered_count, filtered_time)
    }

# Run benchmark
benchmark_results = benchmark_read_methods()

print("\n📈 Performance Summary:")
for method, (count, time_taken) in benchmark_results.items():
    print(f"  {method.capitalize()}: {count} records, {time_taken:.2f}s")

## 8. Incremental Data Loading (CDC Pattern)

In [ ]:
# Simulate incremental data loading pattern
# This is useful for handling large datasets that change over time

def simulate_incremental_load():
    """Demonstrate incremental data loading pattern"""
    
    print("🔄 Incremental Data Loading Simulation:")
    
    # Step 1: Create a tracking table for incremental loads
    from datetime import datetime, timedelta
    
    # Simulate new sales data (representing today's transactions)
    current_timestamp = datetime.now()
    
    new_sales_data = spark.sql("""
        SELECT 
            CONCAT('TXN', LPAD(CAST(ROW_NUMBER() OVER (ORDER BY customer_id) + 100000 AS STRING), 6, '0')) as transaction_id,
            customer_id,
            CURRENT_DATE() as date,
            category,
            product_name,
            CAST(RAND() * 5 + 1 AS INT) as quantity,
            ROUND(unit_price * (0.8 + RAND() * 0.4), 2) as unit_price,
            region
        FROM (
            SELECT DISTINCT 
                customer_id, category, product_name, unit_price, region
            FROM sales_csv 
            ORDER BY RAND()
            LIMIT 50
        )
    """)
    
    # Calculate total amount
    new_sales_data = new_sales_data.withColumn(
        "total_amount", 
        round(col("quantity") * col("unit_price"), 2)
    ).withColumn(
        "created_at",
        current_timestamp
    )
    
    print(f"  📥 Generated {new_sales_data.count()} new transactions")
    
    # Step 2: Write new data to database with timestamp
    try:
        new_sales_data.write \
            .format("jdbc") \
            .option("url", db_properties["url"]) \
            .option("dbtable", "daily_sales_incremental") \
            .option("user", db_properties["user"]) \
            .option("password", db_properties["password"]) \
            .option("driver", db_properties["driver"]) \
            .mode("append") \
            .save()
        
        print("  ✅ New sales data written to incremental table")
        
    except Exception as e:
        print(f"  ❌ Failed to write incremental data: {e}")
    
    # Step 3: Read only new data since last update
    yesterday = current_timestamp - timedelta(days=1)
    
    incremental_query = f"""
        (SELECT * FROM daily_sales_incremental 
         WHERE created_at >= '{yesterday.strftime('%Y-%m-%d %H:%M:%S')}'
        ) as recent_sales
    """
    
    try:
        incremental_data = spark.read \
            .format("jdbc") \
            .option("url", db_properties["url"]) \
            .option("dbtable", incremental_query) \
            .option("user", db_properties["user"]) \
            .option("password", db_properties["password"]) \
            .option("driver", db_properties["driver"]) \
            .load()
        
        print(f"  📊 Retrieved {incremental_data.count()} recent transactions")
        
        # Show sample of incremental data
        incremental_data.select(
            "transaction_id", "customer_id", "category", 
            "total_amount", "created_at"
        ).show(10)
        
    except Exception as e:
        print(f"  ❌ Failed to read incremental data: {e}")

# Run incremental loading simulation
simulate_incremental_load()

## 9. Data Quality and Validation

In [ ]:
# Implement data quality checks between source and target
def run_data_quality_checks():
    """Run comprehensive data quality checks"""
    
    print("🔍 Data Quality Validation:")
    
    # Check 1: Record count consistency
    csv_count = spark.read.csv("/home/jovyan/data/csv/sales_data.csv", header=True).count()
    
    try:
        db_unified_count = spark.read \
            .format("jdbc") \
            .option("url", db_properties["url"]) \
            .option("dbtable", "unified_sales_report") \
            .option("user", db_properties["user"]) \
            .option("password", db_properties["password"]) \
            .option("driver", db_properties["driver"]) \
            .load().count()
        
        print(f"  📊 Record Count Check:")
        print(f"    • Source CSV: {csv_count:,} records")
        print(f"    • Target DB: {db_unified_count:,} records")
        
        if abs(csv_count - db_unified_count) <= 100:  # Allow small variance for filtering
            print(f"    ✅ Record counts are consistent")
        else:
            print(f"    ⚠️ Record count variance: {abs(csv_count - db_unified_count)}")
            
    except Exception as e:
        print(f"    ❌ Could not verify record counts: {e}")
    
    # Check 2: Data type validation
    print(f"\n  🔢 Data Type Validation:")
    
    try:
        db_schema_check = spark.read \
            .format("jdbc") \
            .option("url", db_properties["url"]) \
            .option("dbtable", "customer_analytics") \
            .option("user", db_properties["user"]) \
            .option("password", db_properties["password"]) \
            .option("driver", db_properties["driver"]) \
            .load()
        
        numeric_columns = ['age', 'lifetime_value', 'avg_order_value', 'total_orders']
        
        for col_name in numeric_columns:
            null_count = db_schema_check.filter(col(col_name).isNull()).count()
            if null_count == 0:
                print(f"    ✅ {col_name}: No null values")
            else:
                print(f"    ⚠️ {col_name}: {null_count} null values")
                
    except Exception as e:
        print(f"    ❌ Data type validation failed: {e}")
    
    # Check 3: Business logic validation
    print(f"\n  🎯 Business Logic Validation:")
    
    try:
        business_check_query = """
            (SELECT 
                COUNT(*) as total_records,
                COUNT(CASE WHEN lifetime_value <= 0 THEN 1 END) as negative_ltv,
                COUNT(CASE WHEN age < 18 OR age > 100 THEN 1 END) as invalid_age,
                COUNT(CASE WHEN avg_order_value <= 0 THEN 1 END) as negative_aov
             FROM customer_analytics
            ) as validation_summary
        """
        
        validation_results = spark.read \
            .format("jdbc") \
            .option("url", db_properties["url"]) \
            .option("dbtable", business_check_query) \
            .option("user", db_properties["user"]) \
            .option("password", db_properties["password"]) \
            .option("driver", db_properties["driver"]) \
            .load().collect()[0]
        
        total = validation_results['total_records']
        negative_ltv = validation_results['negative_ltv']
        invalid_age = validation_results['invalid_age']
        negative_aov = validation_results['negative_aov']
        
        print(f"    📊 Total records validated: {total:,}")
        
        if negative_ltv == 0:
            print(f"    ✅ Lifetime values: All positive")
        else:
            print(f"    ⚠️ Lifetime values: {negative_ltv} negative values")
            
        if invalid_age == 0:
            print(f"    ✅ Age values: All within valid range")
        else:
            print(f"    ⚠️ Age values: {invalid_age} outside valid range")
            
        if negative_aov == 0:
            print(f"    ✅ Average order values: All positive")
        else:
            print(f"    ⚠️ Average order values: {negative_aov} negative values")
            
    except Exception as e:
        print(f"    ❌ Business logic validation failed: {e}")
    
    print("\n✅ Data quality validation completed!")

# Run data quality checks
run_data_quality_checks()

## 10. Database Connection Cleanup and Best Practices

In [ ]:
# Demonstrate connection pooling and cleanup best practices
def demonstrate_best_practices():
    """Show database integration best practices"""
    
    print("📋 Database Integration Best Practices:")
    
    print("\n1. 🔐 Connection Security:")
    print("   • Use environment variables for credentials")
    print("   • Implement connection encryption (SSL)")
    print("   • Use connection pooling for better performance")
    
    print("\n2. ⚡ Performance Optimization:")
    print("   • Use partitioned reads for large tables")
    print("   • Set appropriate batchsize for writes")
    print("   • Push filters to database when possible")
    print("   • Use custom queries instead of full table scans")
    
    print("\n3. 🔄 Data Management:")
    print("   • Implement incremental loading patterns")
    print("   • Use appropriate write modes (append, overwrite, ignore)")
    print("   • Implement data quality checks")
    print("   • Monitor and log ETL operations")
    
    print("\n4. 🛠️ Configuration Example:")
    
    optimized_properties = {
        "url": "jdbc:postgresql://postgres:5432/sparkdb?ssl=true",
        "user": "spark",
        "password": "sparkpassword",
        "driver": "org.postgresql.Driver",
        "batchsize": "10000",
        "numPartitions": "8",
        "fetchsize": "10000",
        "queryTimeout": "300",
        "connectionTimeout": "30"
    }
    
    print("   Optimized connection properties:")
    for key, value in optimized_properties.items():
        if key != "password":
            print(f"     {key}: {value}")
        else:
            print(f"     {key}: [HIDDEN]")
    
    print("\n5. 📊 Monitoring Queries:")
    
    # Show current database activity
    try:
        activity_query = """
            (SELECT 
                schemaname,
                tablename,
                n_tup_ins as inserts,
                n_tup_upd as updates,
                n_tup_del as deletes
             FROM pg_stat_user_tables
             WHERE schemaname = 'public'
            ) as table_activity
        """
        
        activity_df = spark.read \
            .format("jdbc") \
            .option("url", db_properties["url"]) \
            .option("dbtable", activity_query) \
            .option("user", db_properties["user"]) \
            .option("password", db_properties["password"]) \
            .option("driver", db_properties["driver"]) \
            .load()
        
        print("\n   Database table activity:")
        activity_df.show()
        
    except Exception as e:
        print(f"   ❌ Could not retrieve activity stats: {e}")

# Run best practices demonstration
demonstrate_best_practices()

# Final summary of created tables
print("\n" + "="*50)
print("📊 SUMMARY OF CREATED DATABASE OBJECTS")
print("="*50)

created_tables = [
    "customer_analytics",
    "product_sales_summary",
    "unified_sales_report",
    "daily_sales_incremental"
]

for table in created_tables:
    try:
        count = spark.read \
            .format("jdbc") \
            .option("url", db_properties["url"]) \
            .option("dbtable", table) \
            .option("user", db_properties["user"]) \
            .option("password", db_properties["password"]) \
            .option("driver", db_properties["driver"]) \
            .load().count()
        
        print(f"✅ {table}: {count:,} records")
        
    except Exception as e:
        print(f"❌ {table}: Error - {e}")

print("\n🎉 Database integration demo completed successfully!")

## Summary

This notebook demonstrated comprehensive database integration capabilities with Apache Spark:

### Key Features Covered:

1. **JDBC Connectivity**: 
   - Basic database connections to PostgreSQL
   - Reading and writing data with various options
   - Custom SQL queries and complex operations

2. **Performance Optimization**:
   - Partitioned reads for parallel processing
   - Batch size optimization for writes
   - Connection pooling and timeout settings
   - Query pushdown and filtering at source

3. **ETL/ELT Patterns**:
   - Multi-source data integration
   - Data transformation and enrichment
   - Unified reporting and analytics
   - Incremental loading strategies

4. **Data Quality & Validation**:
   - Record count consistency checks
   - Data type and schema validation
   - Business logic validation rules
   - Error handling and monitoring

5. **Production Best Practices**:
   - Security and credential management
   - Connection optimization
   - Monitoring and logging
   - Error handling and recovery

### Real-World Applications:

- **Data Warehousing**: ETL pipelines for enterprise data warehouses
- **Real-time Analytics**: Streaming data integration with databases
- **Data Lake Integration**: Combining structured and unstructured data
- **Reporting Systems**: Automated report generation and distribution
- **Data Migration**: Large-scale data movement between systems

### Architecture Patterns:

- **Lambda Architecture**: Batch and real-time processing layers
- **Kappa Architecture**: Stream-first processing approach
- **Data Mesh**: Decentralized data architecture
- **Medallion Architecture**: Bronze, Silver, Gold data layers

### Database Tables Created:

1. `customer_analytics` - Customer lifetime value and behavior metrics
2. `product_sales_summary` - Product performance and profitability analysis
3. `unified_sales_report` - Comprehensive sales reporting with enriched data
4. `daily_sales_incremental` - Incremental data loading demonstration

This integration approach enables scalable, efficient, and reliable data processing workflows that can handle enterprise-scale datasets while maintaining data quality and performance.